In [1]:
import pandas as pd
import sqlite3

# Load CSVs
fuel = pd.read_csv('../data/fuel_prices_1970_2026.csv')
energy = pd.read_csv('../data/owid-energy-data.csv', low_memory=False)

# Quick check
print("Fuel prices:", fuel.shape)
print("Energy data:", energy.shape)
print("\nFuel columns:", list(fuel.columns))
print("\nDate range:", fuel['Date'].min(), "→", fuel['Date'].max())

Fuel prices: (675, 2)
Energy data: (23232, 130)

Fuel columns: ['Date', 'Crude_Oil_Price']

Date range: 1970-01-01 → 2026-03-01


In [2]:
# Connect (creates the file automatically)
conn = sqlite3.connect('../data/energy_capstone.db')

# Prep fuel data — add a year column so we can join with energy data
fuel = pd.read_csv('../data/fuel_prices_1970_2026.csv')
fuel['Date'] = pd.to_datetime(fuel['Date'])
fuel['year'] = fuel['Date'].dt.year

# Aggregate to annual average price
fuel_annual = fuel.groupby('year')['Crude_Oil_Price'].mean().reset_index()
fuel_annual.columns = ['year', 'avg_crude_price']

# Load energy data
energy = pd.read_csv('../data/owid-energy-data.csv', low_memory=False)

# Write both to SQLite
fuel_annual.to_sql('fuel_prices', conn, if_exists='replace', index=False)
energy.to_sql('energy', conn, if_exists='replace', index=False)

print("Database ready!")
print(f"fuel_prices table: {fuel_annual.shape[0]} rows")
print(f"energy table: {energy.shape[0]} rows")

Database ready!
fuel_prices table: 57 rows
energy table: 23232 rows


In [11]:
# Biggest oil price shocks by year — uses LAG to compare year-over-year
# sorting by absolute value catches both spikes and crashes
query1 = """
SELECT 
    year,
    ROUND(avg_crude_price, 2) as price,
    ROUND(avg_crude_price - LAG(avg_crude_price) OVER (ORDER BY year), 2) as change,
    ROUND(
        (avg_crude_price - LAG(avg_crude_price) OVER (ORDER BY year)) 
        / LAG(avg_crude_price) OVER (ORDER BY year) * 100, 1
    ) as pct_change
FROM fuel_prices
ORDER BY ABS(pct_change) DESC
LIMIT 10
"""

pd.read_sql(query1, conn)

,year,price,change,pct_change
0,1974,10.97,8.17,290.9
1,1979,30.96,18.05,139.7
2,2021,69.07,27.81,67.4
3,2000,28.23,10.16,56.2
4,1973,2.81,0.99,54.3
5,2015,50.75,-45.48,-47.3
6,1986,14.35,-12.83,-47.2
7,2005,53.39,15.66,41.5
8,2022,97.10,28.03,40.6
9,1971,1.69,0.48,39.7


In [12]:
# Top 15 countries by renewable share in 2023
# iso_code length = 3 filters out regional aggregates like 'World' and 'Asia'
query2 = """
SELECT 
    country,
    year,
    ROUND(renewables_share_energy, 1) as renewables_pct,
    ROUND(fossil_share_energy, 1) as fossil_pct
FROM energy
WHERE year = 2023
    AND renewables_share_energy IS NOT NULL
    AND iso_code IS NOT NULL
    AND LENGTH(iso_code) = 3
ORDER BY renewables_share_energy DESC
LIMIT 15
"""

pd.read_sql(query2, conn)

,country,year,renewables_pct,fossil_pct
0,Iceland,2023,81.8,18.2
1,Norway,2023,70.8,29.2
2,Sweden,2023,52.5,27.1
3,Brazil,2023,49.0,50.1
4,New Zealand,2023,42.3,57.7
5,Denmark,2023,40.9,59.1
6,Austria,2023,40.0,60.0
7,Finland,2023,35.9,38.5
8,Switzerland,2023,35.6,46.0
9,Portugal,2023,34.7,65.3


In [13]:
# Join oil prices to world energy mix by year
# this is the core of the project — shows if price spikes drove renewable growth
query3 = """
SELECT 
    e.year,
    ROUND(f.avg_crude_price, 2) as oil_price,
    ROUND(e.renewables_share_energy, 2) as world_renewables_pct,
    ROUND(e.solar_share_energy, 2) as solar_pct,
    ROUND(e.wind_share_energy, 2) as wind_pct
FROM energy e
JOIN fuel_prices f ON e.year = f.year
WHERE e.country = 'World'
    AND e.year >= 1970
ORDER BY e.year
"""

df_connection = pd.read_sql(query3, conn)
df_connection

,year,oil_price,world_renewables_pct,solar_pct,wind_pct
0,1970,1.21,5.89,0.00,0.00
1,1971,1.69,5.93,0.00,0.00
2,1972,1.82,5.89,0.00,0.00
3,1973,2.81,5.65,0.00,0.00
4,1974,10.97,6.17,0.00,0.00
5,1975,10.43,6.20,0.00,0.00
6,1976,11.63,5.88,0.00,0.00
7,1977,12.57,5.88,0.00,0.00
8,1978,12.92,6.12,0.00,0.00
9,1979,30.96,6.23,0.00,0.00
